# Osa 1: Tekstigeneraattori

## Kirjastot

In [ ]:
import os
os.environ['KERAS_BACKEND'] = 'tensorflow'  # You can change to 'jax' or 'torch' if preferred

import re
import time
import numpy as np
import pandas as pd
import keras
import sentencepiece as spm
import matplotlib.pyplot as plt

print(f"Keras version: {keras.__version__}")
print(f"Keras backend: {keras.config.backend()}")

## Aineisto

In [ ]:
# Load the Sherlock Holmes text
TEXT_FILE = './sherlock_puhdas.txt'
with open(TEXT_FILE, 'r', encoding='utf-8-sig') as file:
    text = file.read()

print(f"Text length: {len(text)} characters ({len(text.encode('utf-8'))/1024:.0f} KB)")
print(f"First 100 characters: {text[:100]}")

## Tokenisointi

In [ ]:
def train_tokenizer(vocab_size):
    """Train (or load) a SentencePiece BPE tokenizer with the given vocabulary size."""
    os.makedirs("tokenizers", exist_ok=True)
    model_prefix = f"tokenizers/sherlock_sp_{vocab_size}"
    if not os.path.exists(model_prefix + ".model"):
        spm.SentencePieceTrainer.train(
            input=TEXT_FILE,
            model_prefix=model_prefix,
            vocab_size=vocab_size,
            character_coverage=1.0,
            model_type='bpe',
            user_defined_symbols=['<PAD>', '<UNK>'],
            minloglevel=2
        )
    sp = spm.SentencePieceProcessor()
    sp.load(model_prefix + ".model")
    return sp

sp = train_tokenizer(2000)

# Test tokenization
test_text = "Mr. Sherlock Holmes, who was usually very late in the mornings"
print(f"Tokenized example: {sp.encode_as_pieces(test_text)}")
print(f"Vocabulary size: {sp.get_piece_size()}")
print(f"Total tokens in text: {len(sp.encode_as_ids(text))}")

In [ ]:
# Sanaston koon vaikutus: kuinka monta tokenia teksti vaatii ja kuinka pitkiä paloja tulee
rivit = []
for v in [500, 1000, 2000, 4000, 8000]:
    s = train_tokenizer(v)
    n = len(s.encode_as_ids(text))
    rivit.append({"vocab_size": v, "tokeneita": n, "merkkia_per_token": len(text) / n})
pd.DataFrame(rivit)

## Datan valmistelu

In [ ]:
def make_data(sp, seq_length, stride, val_fraction=0.1):
    """Tokenize the text and make (input, target) windows. Target is the input shifted by one token."""
    ids = np.array(sp.encode_as_ids(text), dtype=np.int32)

    # Validation set = last part of the text. Splitting before making windows means
    # no token appears in both sets (neighbouring windows would otherwise overlap).
    split = int((1 - val_fraction) * len(ids))
    train_ids, val_ids = ids[:split], ids[split:]

    def windows(a, step):
        w = np.lib.stride_tricks.sliding_window_view(a, seq_length + 1)[::step]
        return w[:, :-1], w[:, 1:]

    train_x, train_y = windows(train_ids, stride)
    val_x, val_y = windows(val_ids, seq_length)
    return train_x, train_y, val_x, val_y

train_x, train_y, val_x, val_y = make_data(sp, seq_length=64, stride=8)
print(f"Train sequences: {train_x.shape}, validation sequences: {val_x.shape}")

## Mallin määrittely

In [ ]:
def get_positional_encoding(max_len, d_model):
    """Create sinusoidal positional encoding."""
    positions = np.arange(max_len)[:, np.newaxis]
    angles = np.arange(d_model)[np.newaxis, :] / d_model
    angles = 1 / (10000**angles)

    pos_encoding = positions * angles
    pos_encoding[:, 0::2] = np.sin(pos_encoding[:, 0::2])
    pos_encoding[:, 1::2] = np.cos(pos_encoding[:, 1::2])

    return pos_encoding.astype(np.float32)

def build_model(vocab_size, seq_length, embed_dim, num_heads, ff_dim, num_layers, lr=3e-4):
    inputs = keras.Input(shape=(seq_length,))
    embedding_layer = keras.layers.Embedding(vocab_size, embed_dim)(inputs)

    # Add positional encoding
    x = embedding_layer + get_positional_encoding(seq_length, embed_dim)

    # Transformer blocks
    for _ in range(num_layers):
        # Multi-head attention with causal mask
        attention_output = keras.layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=0.1
        )(x, x, use_causal_mask=True)

        # Add & Norm
        x = keras.layers.LayerNormalization(epsilon=1e-6)(x + attention_output)

        # Feed-forward network
        ffn = keras.Sequential([
            keras.layers.Dense(ff_dim, activation="relu"),
            keras.layers.Dense(embed_dim),
            keras.layers.Dropout(0.1)
        ])
        ffn_output = ffn(x)

        # Add & Norm
        x = keras.layers.LayerNormalization(epsilon=1e-6)(x + ffn_output)

    # Final output layer
    outputs = keras.layers.Dense(vocab_size)(x)

    model = keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=['accuracy']
    )
    return model

build_model(vocab_size=2000, seq_length=64, embed_dim=64, num_heads=2, ff_dim=128, num_layers=2).summary()

## Koulutusfunktio

In [ ]:
MAX_EPOCHS = 8
STRIDE = 8        # train windows start every STRIDE tokens (1 = all windows, slowest)
BATCH_SIZE = 128

TULOKSET = {}     # nimi -> koulutuksen tulokset ja malli

def kouluta(nimi, vocab_size=2000, seq_length=64, embed_dim=64, num_heads=2, ff_dim=128,
            num_layers=2, epochs=MAX_EPOCHS, stride=STRIDE, batch_size=BATCH_SIZE):
    keras.backend.clear_session()
    keras.utils.set_random_seed(42)
    sp = train_tokenizer(vocab_size)
    train_x, train_y, val_x, val_y = make_data(sp, seq_length, stride)
    model = build_model(vocab_size, seq_length, embed_dim, num_heads, ff_dim, num_layers)

    print(f"=== {nimi}: {model.count_params()/1e6:.2f} M parameters, {len(train_x)} training sequences ===")
    t0 = time.perf_counter()
    history = model.fit(
        train_x, train_y,
        validation_data=(val_x, val_y),
        batch_size=batch_size,
        epochs=epochs,
        verbose=2,
        callbacks=[
            keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True, verbose=1),
            keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=1),
        ]
    )
    aika = time.perf_counter() - t0

    val_loss = min(history.history["val_loss"])
    TULOKSET[nimi] = {
        "model": model, "sp": sp, "history": history.history, "seq_length": seq_length,
        "rivi": {
            "nimi": nimi, "vocab_size": vocab_size, "seq_length": seq_length, "embed_dim": embed_dim,
            "num_heads": num_heads, "ff_dim": ff_dim, "num_layers": num_layers,
            "parametrit_M": round(model.count_params() / 1e6, 2),
            "epookit": len(history.history["loss"]),
            "aika_min": round(aika / 60, 1),
            "val_loss": round(val_loss, 3),
            "perplexity": round(float(np.exp(val_loss)), 1),
            "val_acc": round(max(history.history["val_accuracy"]), 3),
        }
    }
    return TULOKSET[nimi]["rivi"]

def taulukko():
    return pd.DataFrame([t["rivi"] for t in TULOKSET.values()]).set_index("nimi")

## Kokeet: mallin koko

In [ ]:
KOOT = {
    "minimi": dict(embed_dim=32,  num_heads=2, ff_dim=64,  num_layers=1),
    "pieni":  dict(embed_dim=64,  num_heads=2, ff_dim=128, num_layers=2),
    "keski":  dict(embed_dim=128, num_heads=4, ff_dim=256, num_layers=3),
    "iso":    dict(embed_dim=256, num_heads=8, ff_dim=512, num_layers=4),
}

for nimi, asetukset in KOOT.items():
    kouluta(nimi, **asetukset)

taulukko()

## Kokeet: hyperparametrit

In [ ]:
# Yksi parametri kerrallaan muutettuna "pieni"-mallista
for seq in [32, 128]:
    kouluta(f"pieni_seq{seq}", seq_length=seq, **KOOT["pieni"])

for v in [1000, 4000]:
    kouluta(f"pieni_vocab{v}", vocab_size=v, **KOOT["pieni"])

taulukko()

In [ ]:
kouluta("pieni_1kerros", embed_dim=64, num_heads=2, ff_dim=128, num_layers=1)
kouluta("pieni_4kerrosta", embed_dim=64, num_heads=2, ff_dim=128, num_layers=4)
kouluta("pieni_8paata", embed_dim=64, num_heads=8, ff_dim=128, num_layers=2)

taulukko()

## Koulutuskäyrät

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
for nimi, t in TULOKSET.items():
    ax[0].plot(t["history"]["val_loss"], marker="o", label=nimi)
    ax[1].plot(t["history"]["loss"], marker="o", label=nimi)
ax[0].set_title("Validation loss")
ax[1].set_title("Training loss")
for a in ax:
    a.set_xlabel("Epoch")
    a.set_ylabel("Loss")
    a.grid()
ax[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
df = taulukko()

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
df["val_loss"].plot.bar(ax=ax[0], title="Validation loss")
df["aika_min"].plot.bar(ax=ax[1], title="Training time (min)")
ax[2].scatter(df["parametrit_M"], df["val_loss"])
for nimi, r in df.iterrows():
    ax[2].annotate(nimi, (r["parametrit_M"], r["val_loss"]))
ax[2].set_xlabel("Parameters (M)")
ax[2].set_ylabel("Validation loss")
ax[2].set_title("Size vs. quality")
plt.tight_layout()
plt.show()

df.sort_values("val_loss")

## Tekstin generointi

In [ ]:
def generate_text(model, sp, prompt, seq_length, num_tokens=60, temperature=1.0):
    """
    Generate text autoregressively.

    Args:
        temperature: Controls randomness
            - Low (0.1-0.5): Focused, repetitive
            - Medium (0.7-1.0): Balanced
            - High (1.5+): Creative, chaotic
    """
    pad_id = sp.piece_to_id("<PAD>")
    input_ids = sp.encode_as_ids(prompt)
    generated_ids = list(input_ids)

    if len(input_ids) < seq_length:
        input_ids = [pad_id] * (seq_length - len(input_ids)) + input_ids
    else:
        input_ids = input_ids[-seq_length:]

    for _ in range(num_tokens):
        x = np.array([input_ids])
        logits = np.array(model(x, training=False)[0, -1], dtype=np.float64)
        logits = logits / max(temperature, 1e-3)
        exp_logits = np.exp(logits - np.max(logits))
        probs = exp_logits / np.sum(exp_logits)
        next_token = np.random.choice(len(probs), p=probs)
        generated_ids.append(int(next_token))
        input_ids = input_ids[1:] + [int(next_token)]

    return sp.decode(generated_ids)

def generoi(nimi, prompt, temperature=1.0, num_tokens=60):
    t = TULOKSET[nimi]
    return generate_text(t["model"], t["sp"], prompt, t["seq_length"], num_tokens, temperature)

In [ ]:
# Paras malli = pienin validation loss
PARAS = taulukko()["val_loss"].idxmin()
print("Best model:", PARAS)

PROMPTS = [
    "Sherlock Holmes",
    "\"Watson,\" said Holmes,",
    "It is a capital mistake to theorize",
    "The case was",
]
LAMPOTILAT = [0.2, 0.5, 1.0, 1.5]

np.random.seed(1)
for prompt in PROMPTS:
    print("=" * 100)
    print("PROMPT:", prompt)
    for T in LAMPOTILAT:
        print(f"\n--- temperature {T} ---")
        print(generoi(PARAS, prompt, T))

## Tekstin generointi: mallien vertailu

In [ ]:
# Sama aloitusteksti ja lämpötila, eri mallit
np.random.seed(2)
prompt = "\"Watson,\" said Holmes,"
for nimi in ["minimi", "pieni", "keski", "iso"]:
    r = TULOKSET[nimi]["rivi"]
    print(f"--- {nimi} ({r['parametrit_M']} M, val_loss {r['val_loss']}, {r['aika_min']} min) ---")
    print(generoi(nimi, prompt, 0.8, 80))
    print()

## Toistuvuus lämpötilan funktiona

In [ ]:
def toistuvuus(teksti):
    """Share of repeated word trigrams: 0 = no repetition, 1 = fully repetitive."""
    sanat = teksti.lower().split()
    kolmikot = list(zip(sanat, sanat[1:], sanat[2:]))
    return 1 - len(set(kolmikot)) / max(len(kolmikot), 1)

np.random.seed(3)
rivit = []
for T in [0.2, 0.5, 0.8, 1.0, 1.5, 2.0]:
    tekstit = [generoi(PARAS, p, T, 150) for p in PROMPTS for _ in range(3)]
    rivit.append({"temperature": T, "toistuvuus": np.mean([toistuvuus(t) for t in tekstit])})
tul = pd.DataFrame(rivit).set_index("temperature")
tul.plot(marker="o", figsize=(7, 4), title="Repetition vs. temperature")
plt.ylabel("Repeated trigrams")
plt.grid()
plt.show()
tul

## Tallennus

In [ ]:
TULOKSET[PARAS]["model"].save("sherlock_model.keras")
taulukko().to_csv("osa1_tulokset.csv")
print("Saved sherlock_model.keras and osa1_tulokset.csv")
taulukko().sort_values("val_loss")